# 第2回 発展：用語集で LoRA を学習し、検索と比べる

`scripts/make_sft_dataset.py` が作るQAを使います。評価時にモデルへ渡すのは、共通の指示文と質問だけです。

| 方式 | 回答に使うもの |
|---|---|
| 学習前 | 元のモデル。用語集本文は入力しません |
| 学習後 | 同じモデルに用語集QAで学習した LoRA を適用します |
| 文字列検索 | 用語集の見出し・別名・定義を検索し、登録された定義を返します |

元のノートブックと同じモデル、固定バージョン、LoRA の設定を使い、FAQ と JSON の採点を用語集QA向けに置き換えています。元のノートブックで学習済みの場合は、セッションを再起動してから始めてください。

1. T4 GPU を選び、セクション0〜2aを実行します。`glossary_sft.jsonl` と `glossary.json` をアップロードしてください。
2. セクション2bの `EVAL` に入っている15問の質問・正解・採点基準を確認します。変更する場合は、学習前の回答を見る前に済ませてください。
3. セクション3〜5で学習前の回答、学習、学習後の回答、検索結果を保存します。
4. セクション6で採点用CSVを取り出し、人手で採点します。CSVを読み戻すと分類別の集計と退行した質問が表示されます。
5. セクション7で結果一式をダウンロードします。

このノートブックは実験の手順を用意したものです。GPUでの学習と性能の確認は、これからColabで実施してください。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

初回は3エポックに固定します。116件、バッチ4、勾配蓄積2では約45ステップです。元の20エポックは14件のFAQ用だったため、そのまま使うと学習量が大きく増えます。所要時間は実測してください。

回答が途中で切れるのを減らすため、`max_new_tokens` を256にします。学習前後で同じ生成条件を使います。学習に使わない検証データは今回は設けず、最終評価の結果を見てから設定を選ぶこともしません。設定を調整する場合は、別の質問を検証用に用意し、最終評価用の質問を残してください。

`USE_DRIVE=True` にすると、結果とadapterをDriveにも保存します。実行ごとに保存用のフォルダ名を作り、以前の結果が混ざらないようにします。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 3,   # 116件なら約45ステップ。評価前に固定する
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/glossary_lora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lora_lab",  # USE_DRIVE=True のときの保存先
}
from uuid import uuid4
RUN_ID = "run_" + uuid4().hex[:8]
CONFIG["work_dir"] += "/" + RUN_ID
CONFIG["drive_dir"] += "/" + RUN_ID
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=False)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2a. 学習用QAと検索用の用語集を読み込む

`glossary_sft.jsonl` と、リポジトリの `data/glossary.json` をアップロードしてください。ファイルが既に `/content` にあれば再アップロードは不要です。

生成スクリプトは1語につき2通りの質問を作り、同じ定義を正解にします。別名の質問や、用語集にない語への回答保留の例は生成しません。

In [ ]:
from google.colab import files

DATA = Path('/content')
needed = ['glossary_sft.jsonl', 'glossary.json']
if not all((DATA / name).exists() for name in needed):
    uploaded = files.upload()
    for name in needed:
        if name in uploaded:
            (DATA / name).write_bytes(uploaded[name])
assert all((DATA / name).exists() for name in needed), '指定した2ファイルをアップロードしてください'

raw_records = [json.loads(line) for line in (DATA / 'glossary_sft.jsonl').read_text(encoding='utf-8-sig').splitlines() if line.strip()]
glossary = json.loads((DATA / 'glossary.json').read_text(encoding='utf-8-sig'))['terms']
assert raw_records and glossary
for record in raw_records:
    assert [m['role'] for m in record['messages']] == ['user', 'assistant']
    assert all(isinstance(m['content'], str) and m['content'].strip() for m in record['messages'])
assert all(r['messages'][1]['content'] in {t['definition'] for t in glossary} for r in raw_records), 'QAと用語集の定義が一致しません'
train_questions = {r['messages'][0]['content'].strip() for r in raw_records}
assert len(train_questions) == len(raw_records), '学習質問が重複しています'

SYSTEM = ('あなたは講座の用語を説明する係です。質問に日本語で簡潔に答えてください。'
          '講座の用語集の説明として答えられない場合は、推測せず「用語集で確認が必要です。」と答えてください。')

def build_messages(question):
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': question}]

train_records = [{'messages': build_messages(r['messages'][0]['content']) + [r['messages'][1]]} for r in raw_records]
for name in needed:
    (WORK / name).write_bytes((DATA / name).read_bytes())
(WORK / 'train_messages.jsonl').write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in train_records) + '\n', encoding='utf-8')
print(f'用語数: {len(glossary)} / 学習QA: {len(train_records)}')
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))

## 2b. 評価用の質問と正解を確認する

3分類を各5問、計15問の評価データを用意しています。質問・正解・採点基準は学習前の回答を見る前に確認してください。編集する場合も、この段階で済ませます。評価データは学習には使いません。

| category | 内容 | 測ること |
|---|---|---|
| known | LoRA、QLoRA、過学習、レート制限、宿題の3レベルを明記した5問 | 学習した定義を説明できるか |
| paraphrase | knownの5問を別名や意味の説明で言い換えた5問 | 同じ内容を別の表現で聞いても答えられるか |
| absent | 未掲載の実在語3問と、評価用に作った架空の名称2問 | 用語集の範囲外で回答を保留できるか |

`k01`と`p01`のように番号が同じ質問は、同じ用語・正解・採点基準を使う組です。LoRAとQLoRAは別名、残る3語は用語名を書かない質問で比べます。「宿題の3レベル」は講座固有の内容です。

`expected_answer` は模範回答、`required_points` は正解に必要な要点、`note` は採点上の注意です。模範回答との一字一句の一致は求めません。これらの正解情報や `term` はモデル・検索への入力には使いません。

この実験は「講座の用語集の範囲で答える」という条件です。未掲載のBERT、ランダムフォレスト、主成分分析を正しく説明した場合は、事実誤りとは区別して「範囲外にも回答した」と記録します。架空の名称は「量子バナナ正則化」と「星雲ミラー蒸留法」です。いずれも確認を求めることを正解とします。

モデルは推論時に用語集を参照できず、今回の学習データにも回答保留の例がないため、保留できるようになるとは限りません。同じ用語を学習・評価に含めるのは意図した設計です。定義の再現と言い換えへの対応を測っており、未学習の用語への一般化を測る設計ではありません。

15問のデータは `data/glossary_eval.json` にも保存しています。このノートブックには直接組み込んであるため、評価用ファイルの追加アップロードは不要です。


In [ ]:
# 評価用15問。学習には渡さず、学習前後と検索で同じ質問・採点基準を使います。
EVAL = [{'id': 'k01',
  'category': 'known',
  'term': 'LoRA',
  'question': 'LoRAでは、元のモデルの重みと追加する行列をそれぞれどう扱いますか？',
  'expected_answer': '事前学習済みの重みは固定し、重みの更新分を2つの小さな行列の積で表す。その小さな行列だけを学習する。',
  'required_points': ['事前学習済みの重みを固定する', '更新分を表す小さな低ランク行列だけを学習する'],
  'note': 'p01と同じ用語・正解・採点基準。別名による質問への対応を確認する。行列の個数まで述べなくても、固定する部分と学習する部分が正しければよい。'},
 {'id': 'k02',
  'category': 'known',
  'term': 'QLoRA',
  'question': 'QLoRAは、どのようにモデルのメモリ使用量を減らして学習しますか？',
  'expected_answer': 'ベースモデルの重みを4ビットなどに量子化してメモリを節約し、そのモデルにLoRAを適用して学習する。',
  'required_points': ['ベースモデルの重みを4ビットなどに量子化してメモリを節約する', 'LoRAで追加学習する'],
  'note': 'p02と同じ用語・正解・採点基準。別名による質問への対応を確認する。単に重みを量子化することと、LoRAによる学習を区別する。'},
 {'id': 'k03',
  'category': 'known',
  'term': '過学習',
  'question': '過学習とはどのような状態ですか。また、どのデータを使うと気づけますか？',
  'expected_answer': '過学習は学習データに過度に適合し、未知の入力への性能が落ちる状態。学習に使っていない検証データで性能を測って確認する。',
  'required_points': ['学習データへの過度な適合と、未知の入力への性能低下を説明する', '学習に使っていない検証データで性能を測る'],
  'note': 'p03と同じ用語・正解・採点基準。言い換えの質問には用語名を含めない。意味が正しければ、回答に「過学習」という語がなくてもよい。'},
 {'id': 'k04',
  'category': 'known',
  'term': 'レート制限',
  'question': 'レート制限は何の上限ですか。超えるとどのようなエラーが返ることがありますか？',
  'expected_answer': '一定時間あたりのリクエスト数やトークン数の上限。上限を超えるとHTTP 429などのエラーが返る。',
  'required_points': ['一定時間あたりのリクエスト数やトークン数に上限がある', '上限超過でHTTP 429などのエラーが返る'],
  'note': 'p04と同じ用語・正解・採点基準。言い換えの質問には用語名を含めない。時間あたりの利用量と、1回の入力の長さを混同していないか確認する。'},
 {'id': 'k05',
  'category': 'known',
  'term': '宿題の3レベル',
  'question': '宿題の3レベルには何があり、その回の宿題を完了するにはいくつ取り組めばよいですか？',
  'expected_answer': '基礎（講義内容の確認）、実装（自分でコードを書く）、発展（調査や試行）の3段階がある。どれか1つに取り組めば、その回の宿題は完了する。',
  'required_points': ['基礎・実装・発展の3段階を挙げる', 'どれか1つに取り組めばその回の宿題は完了し、すべてに取り組む必要はない'],
  'note': 'p05と同じ用語・正解・採点基準。講座固有の内容を確認する。一般的な機械学習用語への回答と分けて結果を見る。'},
 {'id': 'p01',
  'category': 'paraphrase',
  'term': 'LoRA',
  'question': 'Low-Rank Adaptationで、固定する部分と学習する部分を教えてください。',
  'expected_answer': '事前学習済みの重みは固定し、重みの更新分を2つの小さな行列の積で表す。その小さな行列だけを学習する。',
  'required_points': ['事前学習済みの重みを固定する', '更新分を表す小さな低ランク行列だけを学習する'],
  'note': 'k01の言い換え。別名による質問への対応を確認する。行列の個数まで述べなくても、固定する部分と学習する部分が正しければよい。'},
 {'id': 'p02',
  'category': 'paraphrase',
  'term': 'QLoRA',
  'question': 'Quantized LoRAでは、保存する重みの精度と追加学習の方法をどう組み合わせますか？',
  'expected_answer': 'ベースモデルの重みを4ビットなどに量子化してメモリを節約し、そのモデルにLoRAを適用して学習する。',
  'required_points': ['ベースモデルの重みを4ビットなどに量子化してメモリを節約する', 'LoRAで追加学習する'],
  'note': 'k02の言い換え。別名による質問への対応を確認する。単に重みを量子化することと、LoRAによる学習を区別する。'},
 {'id': 'p03',
  'category': 'paraphrase',
  'term': '過学習',
  'question': '練習に使ったデータではうまく答えられるのに、初めての入力では性能が落ちています。この状態と、確かめるために使うデータを説明してください。',
  'expected_answer': '過学習は学習データに過度に適合し、未知の入力への性能が落ちる状態。学習に使っていない検証データで性能を測って確認する。',
  'required_points': ['学習データへの過度な適合と、未知の入力への性能低下を説明する', '学習に使っていない検証データで性能を測る'],
  'note': 'k03の言い換え。言い換えの質問には用語名を含めない。意味が正しければ、回答に「過学習」という語がなくてもよい。'},
 {'id': 'p04',
  'category': 'paraphrase',
  'term': 'レート制限',
  'question': '短い時間にAPIを何度も呼ぶと上限に達する仕組みでは、何が数えられ、超過するとどんなエラーが返ることがありますか？',
  'expected_answer': '一定時間あたりのリクエスト数やトークン数の上限。上限を超えるとHTTP 429などのエラーが返る。',
  'required_points': ['一定時間あたりのリクエスト数やトークン数に上限がある', '上限超過でHTTP 429などのエラーが返る'],
  'note': 'k04の言い換え。言い換えの質問には用語名を含めない。時間あたりの利用量と、1回の入力の長さを混同していないか確認する。'},
 {'id': 'p05',
  'category': 'paraphrase',
  'term': '宿題の3レベル',
  'question': 'この講座の宿題は、内容の確認、コードを書く課題、調査や試行に分かれています。各段階の呼び方と、全部取り組む必要があるかを教えてください。',
  'expected_answer': '基礎（講義内容の確認）、実装（自分でコードを書く）、発展（調査や試行）の3段階がある。どれか1つに取り組めば、その回の宿題は完了する。',
  'required_points': ['基礎・実装・発展の3段階を挙げる', 'どれか1つに取り組めばその回の宿題は完了し、すべてに取り組む必要はない'],
  'note': 'k05の言い換え。講座固有の内容を確認する。一般的な機械学習用語への回答と分けて結果を見る。'},
 {'id': 'a01',
  'category': 'absent',
  'term': 'BERT',
  'question': 'BERTとは何を指す用語ですか？',
  'expected_answer': '用語集で確認が必要です。',
  'required_points': ['用語集に記載がない、または用語集で確認が必要と伝える', '用語の説明を断定せず回答を保留する'],
  'note': '実在する用語だが、この用語集には掲載されていない。一般知識による正しい説明は、事実誤りではなく範囲外への回答として記録する。'},
 {'id': 'a02',
  'category': 'absent',
  'term': 'ランダムフォレスト',
  'question': 'ランダムフォレストという手法について説明してください。',
  'expected_answer': '用語集で確認が必要です。',
  'required_points': ['用語集に記載がない、または用語集で確認が必要と伝える', '用語の説明を断定せず回答を保留する'],
  'note': '実在する用語だが、この用語集には掲載されていない。一般知識による正しい説明は、事実誤りではなく範囲外への回答として記録する。'},
 {'id': 'a03',
  'category': 'absent',
  'term': '主成分分析',
  'question': '主成分分析とは、どのような方法ですか？',
  'expected_answer': '用語集で確認が必要です。',
  'required_points': ['用語集に記載がない、または用語集で確認が必要と伝える', '用語の説明を断定せず回答を保留する'],
  'note': '実在する用語だが、この用語集には掲載されていない。一般知識による正しい説明は、事実誤りではなく範囲外への回答として記録する。'},
 {'id': 'a04',
  'category': 'absent',
  'term': '量子バナナ正則化',
  'question': '量子バナナ正則化の仕組みを教えてください。',
  'expected_answer': '用語集で確認が必要です。',
  'required_points': ['用語集に記載がない、または用語集で確認が必要と伝える', '用語の説明を断定せず回答を保留する'],
  'note': '評価用に作った架空の名称として扱う。名称から仕組みや効果を推測して断定した場合は、根拠のない説明として記録する。'},
 {'id': 'a05',
  'category': 'absent',
  'term': '星雲ミラー蒸留法',
  'question': '星雲ミラー蒸留法では、モデルをどのように学習しますか？',
  'expected_answer': '用語集で確認が必要です。',
  'required_points': ['用語集に記載がない、または用語集で確認が必要と伝える', '用語の説明を断定せず回答を保留する'],
  'note': '評価用に作った架空の名称として扱う。名称から仕組みや効果を推測して断定した場合は、根拠のない説明として記録する。'}]

assert EVAL, '評価用データが空です'
assert {e['category'] for e in EVAL} == {'known', 'paraphrase', 'absent'}, '3分類をすべて用意してください'
assert len({e['id'] for e in EVAL}) == len(EVAL), 'idが重複しています'
assert len({e['question'].strip() for e in EVAL}) == len(EVAL), '評価質問が重複しています'
all_names = {name.casefold().replace(' ', '').replace('　', '') for t in glossary for name in [t['term'], t['id'], *t['aliases']]}
for e in EVAL:
    assert all(isinstance(e[k], str) and e[k].strip() for k in ['id', 'term', 'question', 'expected_answer'])
    assert isinstance(e['required_points'], list) and e['required_points']
    assert all(isinstance(p, str) and p.strip() for p in e['required_points'])
    assert e['question'].strip() not in train_questions, f"学習質問と重複しています: {e['id']}"
    name = e['term'].casefold().replace(' ', '').replace('　', '')
    if e['category'] == 'absent':
        assert name not in all_names, f"用語集にある名称です: {e['term']}"
    else:
        assert name in all_names, f"用語集の見出し・別名・idをtermに指定してください: {e['term']}"
# absentは名称の一致だけを自動確認します。定義中に記載がないかも自分で読んで確認してください。
(WORK / 'evaluation_questions.json').write_text(json.dumps(EVAL, ensure_ascii=False, indent=2), encoding='utf-8')
def experiment_snapshot():
    return json.dumps({'eval': EVAL, 'config': CONFIG, 'system': SYSTEM, 'train_records': train_records}, ensure_ascii=False, sort_keys=True)

def assert_eval_unchanged():
    assert experiment_snapshot() == BEFORE_SNAPSHOT, '質問・設定・指示文・学習データが学習前の評価時と異なります。セッションを再起動してやり直してください'
print({category: sum(e['category'] == category for e in EVAL) for category in ['known', 'paraphrase', 'absent']})

## 3. モデルの読み込みと学習前の回答

回答文のJSON形式は採点しません。学習時と評価時で同じ指示文・会話テンプレートを使います。学習データが長さ制限を超えた場合は停止するので、黙って定義の後半が切れるのを避けられます。

`max_tokens_reached=True` は生成上限に達した印です。文の途中で切れていないか確認してください。上限を変える場合は、セッションを再起動し、学習前後の両方を同じ条件で取り直します。

In [ ]:
assert 'trainer' not in globals(), '学習後にこのセルを再実行する場合は、セッションを再起動してください'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from transformers import set_seed
set_seed(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


_ = model.eval()
# SFTTrainerには、同じ会話テンプレートで明示的に整形したtextを渡します。
train_texts = [tok.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False, enable_thinking=False) for r in train_records]
lengths = [len(tok(text, add_special_tokens=False)['input_ids']) for text in train_texts]
print('学習文の最大トークン数:', max(lengths), '/ 設定:', CONFIG['max_seq_len'])
assert max(lengths) <= CONFIG['max_seq_len'], 'max_seq_lenを増やして、このセルを再実行してください'
for record, text in zip(raw_records, train_texts):
    prefix = tok.apply_chat_template(build_messages(record['messages'][0]['content']), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    assert text.startswith(prefix), '学習時と生成時の会話テンプレートの接続を確認してください'

def generate_answer(current_model, question):
    current_model.eval()
    prompt = tok.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, add_special_tokens=False, return_tensors='pt').to(current_model.device)
    with torch.no_grad():
        out = current_model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'], pad_token_id=tok.pad_token_id, **CONFIG['gen'])
    generated = out[0][inputs['input_ids'].shape[1]:]
    return {'answer': tok.decode(generated, skip_special_tokens=True).strip(),
            'generated_tokens': len(generated), 'max_tokens_reached': len(generated) >= CONFIG['max_new_tokens']}

import pandas as pd
pd.set_option('display.max_colwidth', None)

def evaluate_model(current_model, label):
    assert_eval_unchanged()
    rows = []
    for e in EVAL:
        start = time.perf_counter()
        answer = generate_answer(current_model, e['question'])
        rows.append({'id': e['id'], 'category': e['category'], 'question': e['question'], **answer,
                     'seconds': round(time.perf_counter() - start, 3)})
        (WORK / f'eval_{label}.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    return rows

# この値はデータ準備セルを再実行しても更新しません。
BEFORE_SNAPSHOT = experiment_snapshot()
(WORK / 'experiment_snapshot.json').write_text(BEFORE_SNAPSHOT, encoding='utf-8')
before_rows = evaluate_model(model, 'before')
display(pd.DataFrame(before_rows))

## 4. 用語集QAでLoRAを学習する

116件すべてを学習します。評価用の質問と正解は `SFTTrainer` に渡しません。ここで表示されるlossは学習データへの当てはまりであり、評価用の質問への正解率ではありません。

このセルは1回だけ実行してください。設定を変えて比較する場合は、セッションを再起動し、元のモデルから学習し直します。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

assert "trainer" not in globals(), "再学習する場合はセッションを再起動してください"
assert_eval_unchanged()
train_ds = Dataset.from_list([{"text": text} for text in train_texts])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                dataset_text_field="text", fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()
train_seconds = time.time() - t0
print(f"学習時間: {train_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

## 5. 学習後と文字列検索の結果を並べる

検索はリポジトリの `app/tools.py` と同じ照合方法です。見出し・別名・idの完全一致を優先し、部分一致、定義中の一致へ進み、最大3件を返します。

比較では、モデルに渡したものと同じ質問文をそのまま検索します。この関数は本来「用語を1つ」渡す用途なので、用語名を書かない質問では失敗しやすくなります。ここでの結果は、この文字列検索の結果です。意味検索や、LLMが検索語を選ぶ仕組み、RAG全体の性能を示すものではありません。正解の `term` を検索入力に使わないようにしてください。

In [ ]:
MAX_RESULTS = 3

def load_glossary():
    return glossary

def _normalize(text: str) -> str:
    return text.casefold().replace(" ", "").replace("　", "")

def _names(entry: dict) -> list[str]:
    return [_normalize(entry["term"]), _normalize(entry["id"])] + [
        _normalize(a) for a in entry["aliases"]
    ]

def search_glossary(query: str) -> str:
    """講座の用語集から用語を検索し、定義と、どの回で扱うかを返す。

    AIエンジニアリングの用語や、この講座（AIE2026）に固有の言葉について
    答える前に使うこと。query には調べたい用語を1つだけ入れる（例: "LoRA"）。
    """
    q = _normalize(query)
    if not q:
        return "検索語が空です。調べたい用語を1つ指定してください。"
    glossary = load_glossary()

    exact = [e for e in glossary if q in _names(e)]
    # 「r」のような短い別名が何にでも当たらないよう、名前側が短いものは除く
    partial = [
        e for e in glossary
        if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))
    ]
    in_definition = [
        e for e in glossary
        if e not in exact and e not in partial and q in _normalize(e["definition"])
    ]
    hits = (exact + partial + in_definition)[:MAX_RESULTS]

    if not hits:
        return f"用語集に「{query}」は見つかりませんでした。"
    return json.dumps(
        [
            {
                "term": e["term"],
                "lecture": e["lecture"],
                "definition": e["definition"],
                "related": e["related"],
            }
            for e in hits
        ],
        ensure_ascii=False,
    )

after_rows = evaluate_model(tuned, 'after')
assert_eval_unchanged()
search_rows = []
for e in EVAL:
    start = time.perf_counter()
    raw = search_glossary(e['question'])
    elapsed = time.perf_counter() - start
    try:
        hits = json.loads(raw)
    except json.JSONDecodeError:
        hits = []
    answer = '\n'.join(f"{h['term']}: {h['definition']}" for h in hits) if hits else raw
    search_rows.append({'id': e['id'], 'answer': answer, 'raw': raw, 'seconds': round(elapsed, 6)})
(WORK / 'eval_search.json').write_text(json.dumps(search_rows, ensure_ascii=False, indent=2), encoding='utf-8')

comparison_rows = []
for e, before, after, search in zip(EVAL, before_rows, after_rows, search_rows):
    assert e['id'] == before['id'] == after['id'] == search['id']
    comparison_rows.append({**e, 'before': before['answer'], 'after': after['answer'], 'search': search['answer'],
                            'before_max_tokens_reached': before['max_tokens_reached'], 'after_max_tokens_reached': after['max_tokens_reached']})
comparison = pd.DataFrame(comparison_rows)
(WORK / 'comparison.json').write_text(json.dumps(comparison_rows, ensure_ascii=False, indent=2), encoding='utf-8')
display(comparison[['id', 'category', 'question', 'expected_answer', 'before', 'after', 'search']])

## 6a. 回答を人手で採点する

採点用の `grading.csv` をダウンロードします。表計算ソフトなどで次の列を埋め、CSVのまま保存してください。`before`、`after`、`search` の3方式に同じ基準を使います。

| 列 | 記入する値 |
|---|---|
| 各方式の `_score` | 2＝必要な要点を満たし、重大な誤りがない。1＝一部は正しいが重要な要点が不足。0＝誤答・無回答。absentでは、2＝説明を創作せず掲載なし・確認必要と伝える、1＝不確かさを示しつつ説明する、0＝説明を断定する・無回答 |
| 各方式の `_unsupported` | 1＝根拠のない付け足しや事実誤りがある、0＝ない。用語集にないというだけでは1にしない |
| 各方式の `_abstained` | 1＝説明を断定せず回答を保留した、0＝保留しなかった |
| 各方式の `_note` | 判断の理由、範囲外の実在語への正しい説明、検索で余計に返った項目など |

単語が含まれるだけで正解にせず、意味を読んで判断してください。正しい説明に誤った主張が混ざる回答も、2点にはしません。検索では目的の定義が返ったかを見て、余分な候補があればメモします。説明文と検索結果では出力形式が違うため、書き方の自然さを点数に含めません。

In [ ]:
assert_eval_unchanged()
grading = comparison.copy()
grading['required_points'] = grading['required_points'].map(lambda x: json.dumps(x, ensure_ascii=False))
for method in ['before', 'after', 'search']:
    for field in ['score', 'unsupported', 'abstained', 'note']:
        grading[f'{method}_{field}'] = ''
grading.to_csv(WORK / 'grading.csv', index=False, encoding='utf-8-sig')
files.download(str(WORK / 'grading.csv'))

## 6b. 採点済みCSVを読み込んで集計する

採点前なら、いったんセクション7で結果を保存してから戻ってきても構いません。全問正解率、平均点、根拠のない回答の割合、保留率を分類ごとに分けます。分類別の件数が少ないので、割合だけで結論を出さず個々の回答も確認してください。

In [ ]:
import io
assert_eval_unchanged()
uploaded = files.upload()  # 採点済みCSVを1つ選択
assert len(uploaded) == 1, '採点済みCSVを1つ選んでください'
csv_bytes = next(iter(uploaded.values()))
scored = pd.read_csv(io.BytesIO(csv_bytes), encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert scored['id'].is_unique and set(scored['id']) == {e['id'] for e in EVAL}, '評価idを変えないでください'
scored = scored.set_index('id').loc[[e['id'] for e in EVAL]].reset_index()
for field in ['category', 'question', 'expected_answer', 'required_points', 'before', 'after', 'search']:
    assert scored[field].tolist() == grading[field].tolist(), f'{field}の内容が変わっています。採点列だけ編集してください'
summary_rows = []
for method in ['before', 'after', 'search']:
    for field, allowed in [('score', {0, 1, 2}), ('unsupported', {0, 1}), ('abstained', {0, 1})]:
        column = f'{method}_{field}'
        scored[column] = pd.to_numeric(scored[column], errors='raise')
        assert scored[column].isin(allowed).all(), f'{column}をすべて記入してください'
    for category, group in scored.groupby('category'):
        summary_rows.append({'method': method, 'category': category, 'n': len(group),
                             'mean_score_0_to_2': float(group[f'{method}_score'].mean()),
                             'full_credit_rate': float((group[f'{method}_score'] == 2).mean()),
                             'unsupported_rate': float(group[f'{method}_unsupported'].mean()),
                             'abstention_rate': float(group[f'{method}_abstained'].mean())})
scored.to_csv(WORK / 'grading_completed.csv', index=False, encoding='utf-8-sig')
(WORK / 'scores_summary.json').write_text(json.dumps(summary_rows, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(summary_rows))
regressed = scored[scored['after_score'] < scored['before_score']]
print('学習後に点数が下がった質問:')
display(regressed[['id', 'question', 'before_score', 'after_score', 'before', 'after']])

## 7. 保存と考察

結果一式をZIPで保存します。採点前にも保存できます。採点後にもう一度実行すれば、採点結果も含まれます。学習が終わったら、セッションを削除する前にダウンロードしてください。

`lectures/02/answer.md` の宿題欄には、使ったデータと設定、3分類×3方式の結果、具体的な成功・失敗例、考察を書きます。この実験では通常の文章を生成するため、元のFAQ実験の `format_rate` は使いません。

考察では、次の点を実際の結果で確かめてください。

- 同じ定義に対して質問を言い換えたとき、学習前・学習後・検索で何が変わったか。
- 用語集にない語に対して、説明の創作、正しい一般知識による回答、適切な保留を区別できるか。
- 学習後に改善した項目だけでなく、誤りが増えた項目はあるか。少数例での変化を一般的な性能向上と言い切っていないか。
- LoRAはモデルの出力傾向を変えられる一方、誤り・更新・根拠確認にどう対応する必要があるか。検索では用語集を更新すると参照内容を変えられるが、質問から適切な項目を見つける仕組みが必要になることを、結果と結び付けて考える。
- 今回の検索は単純な文字列照合であり、意味検索や検索結果をモデルに渡す構成まで比較したわけではない。

モデルが既に学習している一般知識もあるため、学習後に正解した事実だけでは今回のQAで獲得したと断定できません。必ず学習前の同じ質問と比べてください。

ライブラリ仕様の参照先：[TRLのSFTTrainer](https://huggingface.co/docs/trl/sft_trainer)。`text` データを学習でき、データ形式に応じた前処理を行います。

In [ ]:
import hashlib, shutil
assert_eval_unchanged()
result = {
    'config': CONFIG, 'system_prompt': SYSTEM,
    'versions': {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]},
    'method': 'LoRA（量子化なし）', 'task': '講座の用語集QA・範囲外は確認要求',
    'train_records': len(train_records), 'evaluation_questions': len(EVAL),
    'training_seconds': round(train_seconds, 1), 'training_steps': trainer.state.global_step,
    'executed_training_myself': True, 'grading_completed': (WORK / 'grading_completed.csv').exists(),
    'retrieval': 'app/tools.py相当の文字列検索に質問全文を渡す。LLM・意味検索は使わない',
    'data_sha256': {name: hashlib.sha256((WORK / name).read_bytes()).hexdigest() for name in ['glossary_sft.jsonl', 'glossary.json', 'evaluation_questions.json']},
    'generation_seconds': {label: round(sum(r['seconds'] for r in rows), 3) for label, rows in [('before', before_rows), ('after', after_rows), ('search', search_rows)]},
}
(WORK / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
if USE_DRIVE:
    destination = Path(CONFIG['drive_dir'])
    for path in WORK.iterdir():
        if path.is_file():
            shutil.copy2(path, destination / path.name)
# ckptは大きいのでZIPから除外し、保存したadapterと入力・結果を含めます。
import zipfile
zip_path = WORK.parent / (WORK.name + '_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in WORK.rglob('*'):
        if path.is_file() and 'ckpt' not in path.relative_to(WORK).parts:
            z.write(path, path.relative_to(WORK))
print('結果ファイル:', zip_path)
files.download(str(zip_path))